In [8]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

import sys
import os

sys.path.append('../features')
from data_cleaner import DataCleaner
from feature_engineer import FeatureEngineer
from model import AnomalyDetector
from analysis import ClusterInvestigator

In [2]:
cleaner = DataCleaner('../data/diem_thi_THPTQG_2026.csv')
df_raw = cleaner.load_data()
df_cleaned = cleaner.score_under_1() 

df_valid = df_cleaned[df_cleaned['score_under_1'] == False].copy()

engineer = FeatureEngineer(df_valid)
X_matrix = engineer.extract_features()

print(X_matrix.head())

Eplore 581 students under 1 point!
[1/4] Calculating internal feature F1 - Personal std...
[2/4] Calculating internal feature F2: - Z-score range (Max-Min Z-score Range)...
[3/4] Calculation F3: Space fature - Z-score Exam room (Micro-Anomaly)...
[4/4] Calculation F4: Space feature - Z-score Exam cluster/Provice (Macro-Anomaly)...
Finish!
Matrix X is ready: 1,131,394 student, 4 feature (0 NaN).
        SBD  F1_Personal_Std  F2_Z_Range  F3_Micro_Z  F4_Macro_Z
0  01000001         1.760386    2.292650    0.169198    1.019023
1  01000002         1.647726    1.659379    0.059700    0.961489
2  01000003         0.630311    0.937467    0.053146    0.681991
3  01000004         1.447052    1.785315   -1.157063    0.765691
4  01000005         0.239357    0.660813    0.642908    0.722907


In [3]:
detector = AnomalyDetector(contamination=0.001)
df_results = detector.fit_predict(X_matrix)
blacklist_df = detector.extract_blacklist()

In [4]:
print(blacklist_df['Insight_Pattern'].value_counts())

suspects = blacklist_df[blacklist_df['Insight_Pattern'].str.contains("fraud")].head(10)

suspects[['SBD', 'anomaly_score', 'Insight_Pattern', 'F1_Personal_Std_scaled', 'F3_Micro_Z_scaled', 'F4_Macro_Z_scaled']]

Insight_Pattern
Pattern 1: ....                                      655
Pattern 0: Mixed anomaly                             328
Pattern 3: Organized fraud (Exam cluster/Provice)    116
Pattern 2: Personal fraud                             33
Name: count, dtype: int64


,SBD,anomaly_score,Insight_Pattern,F1_Personal_Std_scaled,F3_Micro_Z_scaled,F4_Macro_Z_scaled
151659,11001228,-0.077894,Pattern 3: Organized fraud (Exam cluster/Provice),3.245940,-2.124501,3.428588
148664,08016418,-0.073402,Pattern 3: Organized fraud (Exam cluster/Provice),3.138075,-1.684478,3.894431
148816,08016571,-0.071364,Pattern 3: Organized fraud (Exam cluster/Provice),3.014244,-2.649542,3.816948
398578,33000261,-0.069719,Pattern 3: Organized fraud (Exam cluster/Provice),3.299600,-1.414369,3.951031
187133,15009997,-0.067118,Pattern 3: Organized fraud (Exam cluster/Provice),3.834573,-1.611980,3.631732
167320,14004319,-0.066939,Pattern 3: Organized fraud (Exam cluster/Provice),2.817289,0.708959,4.051923
492290,38002628,-0.066331,Pattern 3: Organized fraud (Exam cluster/Provice),3.539358,-1.894972,3.208749
746083,66002006,-0.065773,Pattern 2: Personal fraud,3.848960,2.700940,-1.812147
187387,15010251,-0.065056,Pattern 3: Organized fraud (Exam cluster/Provice),3.514314,-1.547638,4.080752
187321,15010185,-0.065016,Pattern 3: Organized fraud (Exam cluster/Provice),3.271532,-1.437195,3.919545


In [5]:
from IPython import display
import json
target_sbds = [11001228, 8016418, 8016571, 33000261, 15009997, 
               14004319, 38002628, 66002006, 15010251, 15010185]

output_dir = '../results'
os.makedirs(output_dir, exist_ok=True)

df_raw['SBD_int'] = df_raw['SBD'].astype(int)
columns_to_show = ['SBD', 'Toán', 'Văn', 'Lý', 'Hóa', 'Sinh', 'Sử', 
                'Địa', 'GD Kinh tế - Pháp luật', 'Tin học', 
                'Công nghệ', 'Ngoại ngữ']

investigation_data = {}
for target_sbd in target_sbds:
    room_suspects = df_raw[(df_raw['SBD_int'] >= target_sbd - 2) & 
                           (df_raw['SBD_int'] <= target_sbd + 3)].copy()

    print(f"Detect of SBD: {target_sbd}")
    print(room_suspects[columns_to_show].to_string(index=False))

    records = room_suspects[columns_to_show].to_dict(orient='records')
    
    investigation_data[str(target_sbd)] = records

json_path = f"{output_dir}/anomaly_detector_result.json"

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(
        investigation_data, 
        f, 
        ensure_ascii=False, # Giữ nguyên font chữ tiếng Việt (Toán, Lý, Hóa...)
        indent=4            # Thụt lề 4 space cho file json đẹp và dễ đọc
    )

Detect of SBD: 11001228
     SBD  Toán  Văn   Lý  Hóa  Sinh   Sử  Địa  GD Kinh tế - Pháp luật  Tin học  Công nghệ  Ngoại ngữ
11001226   7.0 5.25 4.20 4.25   NaN  NaN  NaN                     NaN      NaN        NaN        NaN
11001227   9.0 7.50 8.25 8.75   NaN  NaN  NaN                     NaN      NaN        NaN        NaN
11001228   5.0 9.00  NaN  NaN   NaN  NaN  NaN                     4.6      NaN        NaN        2.0
11001229   9.0 6.00 9.00 9.00   NaN  NaN  NaN                     NaN      NaN        NaN        NaN
11001230   8.0 8.50  NaN  NaN   NaN 9.75  NaN                     NaN      NaN        NaN        9.0
11001231   7.5 8.00  NaN  NaN  5.75  NaN  NaN                     NaN      NaN        NaN        6.0
Detect of SBD: 8016418
     SBD  Toán  Văn  Lý  Hóa  Sinh   Sử  Địa  GD Kinh tế - Pháp luật  Tin học  Công nghệ  Ngoại ngữ
08016416  9.50 8.75 NaN  NaN   NaN 10.0  8.1                     NaN      NaN        NaN        NaN
08016417 10.00 9.00 8.5  NaN   NaN  NaN  NaN  

In [6]:
# target_sbd = 11001228
# df_raw['SBD_int'] = df_raw['SBD'].astype(int)

# room_suspects = df_raw[(df_raw['SBD_int'] >= target_sbd - 2) & 
#                        (df_raw['SBD_int'] <= target_sbd + 3)]

# columns_to_show = ['SBD', 'Toán', 'Lý', 'Hóa', 'Văn', 'Ngoại ngữ']

# print(f"Raw score of {target_sbd} and next 5 students:")
# display(room_suspects[columns_to_show].style.background_gradient(cmap='Reds'))
from IPython import display

pattern3_df = blacklist_df[blacklist_df['Insight_Pattern'].str.contains("Pattern 3")].copy()
pattern3_df['Province_Code'] = pattern3_df['SBD'].astype(str).str.zfill(8).str[:2]

anomaly_clusters = pattern3_df.groupby('Province_Code').size().reset_index(name='Number_of_Anomaly')

anomaly_clusters = anomaly_clusters.sort_values(by='Number_of_Anomaly', ascending=False).reset_index(drop=True)
anomaly_clusters.head(10).style.bar(subset=['Number_of_Anomaly'], color='#d65f5f')


,Province_Code,Number_of_Anomaly
0,33,15
1,14,13
2,86,13
3,38,12
4,15,11
5,25,11
6,75,8
7,37,6
8,08,5
9,79,4


In [7]:
investigator = ClusterInvestigator(pattern3_df)
Pro_code_33_results = investigator.investigate_province(province_code='33', n_clusters=2)

cluster_counts = Pro_code_33_results['Cluster'].value_counts()
print("\nDistribution anomalies by cluster:")
print(cluster_counts.to_string())

largest_cluster = cluster_counts.idxmax()
root = Pro_code_33_results[Pro_code_33_results['Cluster'] == largest_cluster]

print(f"Including {len(root)} students with strong relationship.")

min_seq = root['SBD_seq'].min()
max_seq = root['SBD_seq'].max()
kc = max_seq - min_seq

print(f"SBD series: 33{min_seq:06d} to 33{max_seq:06d}")
print(f"Distance between firt student and last student: {kc} number (appr {(kc//24) + 1} room).")
print(root['SBD'].tolist())


Distribution anomalies by cluster:
Cluster
0    10
1     5
Including 10 students with strong relationship.
SBD series: 33000176 to 33000370
Distance between firt student and last student: 194 number (appr 9 room).
['33000261', '33000306', '33000228', '33000316', '33000370', '33000327', '33000282', '33000190', '33000355', '33000176']


In [ ]:
X_1d = Pro_code_33_results[['SBD_seq']].values
sse = []
silhouette_scores = []

K_range = range(2, min(7, len(X_1d)))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(X_1d)
        
        sse.append(kmeans.inertia_)
        
        score = silhouette_score(X_1d, labels)
        silhouette_scores.append(score)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel('Number of cluster (K)', fontsize=12, fontweight='bold')
ax1.set_ylabel('SSE (Inertia)', color='tab:red', fontsize=12, fontweight='bold')
ax1.plot(K_range, sse, marker='o', color='tab:red', linewidth=2, markersize=8, label='SSE (Elbow)')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2 = ax1.twinx() 
ax2.set_ylabel('Silhouette Score', color='tab:blue', fontsize=12, fontweight='bold')
ax2.plot(K_range, silhouette_scores, marker='s', color='tab:blue', linewidth=2, markersize=8, label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.axvline(x=2, color='black', linestyle='--', alpha=0.5, label='Optimal K (Elbow)')
plt.title('Evaluate K-MEANS: ELBOW METHOD & SILHOUETTE SCORE', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()
